# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [16]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _line item on one order, containing vendor info_

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [17]:
import pandas as pd
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [18]:
# TODO
joined = lines.merge(
    pd.json_normalize(vendors_json),
    on='vendor_id',
    how='left',
    indicator=True
)

unmatched = joined[joined['_merge'] == 'left_only']
print('unmatched vendor id:', unmatched['vendor_id'].unique())
print('units attached:', unmatched['qty'].sum())

joined

unmatched vendor id: ['V-07']
units attached: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [19]:
# TODO
joined['name'] = joined['name'].fillna('Unknown vendor (V-07)')
by_vendor = joined.groupby('name')['qty'].sum().sort_values(ascending=False)

# 3 units belong to V-07, which isn't in the vendor list, dropping
# that row would understand total units by 3, leaving name NaN
# would make it disappear without warning, so we should label it
# and flag which vendor needs to be added.

by_vendor

,qty
name,
Hoos Burgers,3
Unknown vendor (V-07),3
Cav Merch,1


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [20]:
# TODO
by_zone = joined.groupby('zone')['qty'].sum().sort_values(ascending=False)
print(by_zone)
print('total:', by_zone.sum())
print('difference from 7:', 7 - by_zone.sum())

zone
A    4
Name: qty, dtype: int64
total: 4
difference from 7: 3


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [21]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
ragged_fixed = [
    {**order, 'lines': order.get('lines', [])}
    for order in ragged_json
]

lines_ragged = pd.json_normalize(
    ragged_fixed,
    record_path='lines',
    meta=['order', 'vendor_id']
)

lines_ragged

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [22]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per item makes counting easy, just groupby('item') and ['qty'].sum(). With one row per order you'd have to dig into each order's lines list manually to add up how many of one item sold across orders. Flattening does that work for you.

b) Missing quantity (NaN) means we don't know how many fries sold, qty = 0 means we know for sure none sold. If you fillna(0) you're pretending there is data that we don't have, and totals come out too low where you can't tell "didn't sell" from "wasn't recorded." It's the same as: None != x=0.